In [3]:
!pip install -q langchain langchain-groq

In [4]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=key,
    temperature=0.3
)

print("Groq connection setup completed!")

Groq connection setup completed!


In [18]:
SYSTEM = """
ROLE:
You are the friendly customer support assistant
for Sweet Crumbs Home Bakery.

BUSINESS FACTS:
1. Business name: Sweet Crumbs Home Bakery.
2. Plum cake costs ₹500.
3. Chocolate cake costs ₹650 and cupcakes cost ₹80 each.
4. Eggless cake options are available.
5. All orders require at least 24 hours' advance notice.

RULES:
1. Be polite and concise. Keep replies under 50 words.
2. Never invent products, prices, discounts or business policies.
3. Never promise unconfirmed delivery times.
4. When a customer asks for the price of multiple items,
   calculate the total using the listed menu prices.
   If information is unavailable, say:
   "I'll check with the owner."
"""

In [19]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [7]:
print (ask("Do you make eggless cakes?"))

Yes, we offer eggless cake options. Let us know which flavor you’d like!


In [8]:
print(ask("Do you make eggless cakes?"))

print(ask("Can I order a cake for today evening?"))

print(ask("Can you guarantee delivery at 5 PM?"))

Yes, we offer eggless cake options. Let us know which flavor you’d like!
I’m sorry, but we require at least 24 hours’ notice for all orders, so we can’t fulfill a same‑day request. Please let us know a future date, and we’ll be happy to arrange it!
I’ll check with the owner about that delivery time and get back to you.


In [9]:
%%writefile /content/tests.csv
question,must_include
What is the price of plum cake?,500
How much does chocolate cake cost?,650
What is the price of a cupcake?,80
Do you make eggless cakes?,eggless
How much advance notice is required?,24 hours
Can I order a cake with only 2 hours notice?,24 hours
Can you guarantee delivery at 5 PM?,check with the owner
Do you deliver to Amalapuram?,check with the owner
Do you offer a bulk order discount?,check with the owner
Do you have other cake flavours?,check with the owner

Writing /content/tests.csv


In [10]:
import csv

def run_tests(path="/content/tests.csv"):
    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        success = expected in answer.lower()

        if success:
            passed += 1

        print(
            "PASS" if success else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:100]
        )

    percentage = passed / len(rows) * 100

    print(f"\n{passed}/{len(rows)} passed ({percentage:.0f}%)")

    return passed, percentage

In [11]:
baseline_result = run_tests()

PASS | What is the price of plum cake? -> The plum cake is priced at ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake costs ₹650.
PASS | What is the price of a cupcake? -> A cupcake costs ₹80 each.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like!
FAIL | How much advance notice is required? -> All orders need at least 24 hours’ advance notice.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Please let us know a day ah
PASS | Can you guarantee delivery at 5 PM? -> I’m sorry, I can’t guarantee a specific delivery time. I’ll check with the owner and get back to you
PASS | Do you deliver to Amalapuram? -> I'll check with the owner and get back to you.
PASS | Do you offer a bulk order discount? -> I'll check with the owner.
FAIL | Do you have other cake flavours? -> We currently offer plum cake and chocolate cake (both also in eggle

In [13]:
temperature_result_1 = run_tests()

PASS | What is the price of plum cake? -> The plum cake is priced at ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake costs ₹650.
PASS | What is the price of a cupcake? -> A cupcake costs ₹80 each.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like!
FAIL | How much advance notice is required? -> We require at least 24 hours’ advance notice for all orders.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Please let us know a day ah
PASS | Can you guarantee delivery at 5 PM? -> I’m sorry, I can’t guarantee a specific delivery time. I’ll check with the owner and let you know.
PASS | Do you deliver to Amalapuram? -> I'll check with the owner.
PASS | Do you offer a bulk order discount? -> We don’t have a listed bulk‑order discount at the moment. I’ll check with the owner and let you know
FAIL | Do you have other cake flavours? -> We cu

In [14]:
temperature_result_2 = run_tests()

PASS | What is the price of plum cake? -> The plum cake costs ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake costs ₹650.
PASS | What is the price of a cupcake? -> A cupcake costs ₹80.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like, and we’ll prepare it for yo
FAIL | How much advance notice is required? -> We require at least 24 hours’ advance notice for all orders.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Let us know when you’d like
PASS | Can you guarantee delivery at 5 PM? -> I’m sorry, I can’t guarantee a 5 PM delivery. I’ll check with the owner.
PASS | Do you deliver to Amalapuram? -> I'll check with the owner.
PASS | Do you offer a bulk order discount? -> I’m not sure about bulk‑order discounts. I’ll check with the owner and get back to you.
FAIL | Do you have other cake flavours? -> We currently offer plum cake

In [17]:
print("Temperature 1.0 - Run 1:", temperature_result_1[1], "%")
print("Temperature 1.0 - Run 2:", temperature_result_2[1], "%")

Temperature 1.0 - Run 1: 70.0 %
Temperature 1.0 - Run 2: 70.0 %


In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

In [21]:
improved_result = run_tests()

PASS | What is the price of plum cake? -> The plum cake is priced at ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake costs ₹650.
PASS | What is the price of a cupcake? -> A cupcake costs ₹80 each.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like, and we’ll prepare it for yo
FAIL | How much advance notice is required? -> We require at least 24 hours’ advance notice for all orders.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Please let us know when you
FAIL | Can you guarantee delivery at 5 PM? -> We can’t guarantee a specific delivery time, but we’ll do our best to have it there by 5 PM. Please 
PASS | Do you deliver to Amalapuram? -> I'll check with the owner and get back to you about delivery to Amalapuram.
PASS | Do you offer a bulk order discount? -> We don’t have a bulk‑order discount listed at the moment. I’ll check w

In [22]:
print("Before improvement:", baseline_result[1], "%")
print("After improvement:", improved_result[1], "%")

Before improvement: 70.0 %
After improvement: 70.0 %


In [23]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=key,
    temperature=1.0
)

In [24]:
chain = prompt | llm | StrOutputParser()

In [25]:
temperature_result_1 = run_tests()

PASS | What is the price of plum cake? -> The plum cake is priced at ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake is priced at ₹650.
PASS | What is the price of a cupcake? -> A cupcake costs ₹80 each.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like, and we’ll prepare it for yo
FAIL | How much advance notice is required? -> All orders need at least 24 hours’ advance notice.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Let us know when you’d like
FAIL | Can you guarantee delivery at 5 PM? -> I’m sorry, we can’t guarantee a specific delivery time such as 5 PM. We aim to deliver within our us
PASS | Do you deliver to Amalapuram? -> I'll check with the owner regarding delivery to Amalapuram and get back to you shortly.
PASS | Do you offer a bulk order discount? -> We don’t have a bulk‑order discount listed at the moment. I’l

In [26]:
temperature_result_2 = run_tests()

PASS | What is the price of plum cake? -> The plum cake is priced at ₹500.
PASS | How much does chocolate cake cost? -> The chocolate cake costs ₹650.
PASS | What is the price of a cupcake? -> Each cupcake is ₹80.
PASS | Do you make eggless cakes? -> Yes, we offer eggless cake options. Let us know which flavor you’d like, and we’ll prepare it for yo
FAIL | How much advance notice is required? -> We require at least 24 hours' advance notice for all orders.
FAIL | Can I order a cake with only 2 hours notice? -> I’m sorry, but we require at least 24 hours’ notice for all cake orders. Let us know a day ahead, an
FAIL | Can you guarantee delivery at 5 PM? -> We can’t guarantee a specific 5 PM delivery, but we’ll do our best to get your order to you as close
PASS | Do you deliver to Amalapuram? -> I'll check with the owner about delivery to Amalapuram and get back to you shortly.
PASS | Do you offer a bulk order discount? -> We don’t have a bulk‑order discount listed at the moment. I’ll chec

In [27]:
print("Temperature 1.0 - Run 1:", temperature_result_1[1], "%")
print("Temperature 1.0 - Run 2:", temperature_result_2[1], "%")

Temperature 1.0 - Run 1: 60.0 %
Temperature 1.0 - Run 2: 60.0 %


In [28]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=key,
    temperature=0.3
)

chain = prompt | llm | StrOutputParser()

In [29]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, quantity: int = 1) -> int:
    """Return the total price of a bakery item."""

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not found in menu.")

    if quantity <= 0:
        raise ValueError("Quantity must be greater than zero.")

    return MENU[item] * quantity

In [30]:
print(get_price("plum cake"))
print(get_price("cupcake", 6))
print(get_price("chocolate cake", 2))

500
480
1300


In [31]:
def check_order_availability(hours_until_order: int) -> bool:
    """Check whether an order meets the 24-hour advance notice requirement."""

    return hours_until_order >= 24

In [32]:
print(check_order_availability(48))
print(check_order_availability(24))
print(check_order_availability(5))

True
True
False
